In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import re

In [2]:
df = pd.read_csv('../data/preprocessed_data/news_data.csv')
print(df.shape)
df["date"]=pd.to_datetime(df["date"]).dt.normalize()
df.head()


(675, 11)


,company,ticker,text,created,nid,article_title,article_url,info_index,datetime,datetime_my,date
0,Petronas Chemicals,5183.KL,PETRONAS Chemicals Group Bhd (KL: PCHEM ) rose...,1789397619000,818088,"Kelington, NuEnergy, GFM, Exsim Hospitality, K...",https://theedgemalaysia.com/node/818088,6,2026-09-14 14:53:39+00:00,2026-09-14 22:53:39+08:00,2026-09-14
1,Maybank,1155.KL,Malayan Banking Bhd (KL: MAYBANK ) has receive...,1788879258000,817315,"Public Bank, Maybank, Hektar REIT, Cuscapi, He...",https://theedgemalaysia.com/node/817315,2,2026-09-08 14:54:18+00:00,2026-09-08 22:54:18+08:00,2026-09-08
2,Public Bank,1295.KL,Public Bank Bhd (KL: PBBANK ) is proposing to ...,1788879258000,817315,"Public Bank, Maybank, Hektar REIT, Cuscapi, He...",https://theedgemalaysia.com/node/817315,1,2026-09-08 14:54:18+00:00,2026-09-08 22:54:18+08:00,2026-09-08
3,CIMB,1023.KL,CIMB Group Holdings Bhd ’s (KL: CIMB ) net pro...,1787932174000,816234,"Yinson, Gamuda, WCT, Zetrix AI, HeiTech Padu, ...",https://theedgemalaysia.com/node/816234,6,2026-08-28 15:49:34+00:00,2026-08-28 23:49:34+08:00,2026-08-28
4,Tenaga Nasional,5347.KL,Tenaga Nasional Bhd (KL: TENAGA ) reported a 2...,1787848535000,816073,"Tenaga Nasional, Inari Amertron, Sunway, Berja...",https://theedgemalaysia.com/node/816073,1,2026-08-27 16:35:35+00:00,2026-08-28 00:35:35+08:00,2026-08-28


In [3]:
display(df.info())
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 675 entries, 0 to 674
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   company        675 non-null    object        
 1   ticker         675 non-null    object        
 2   text           675 non-null    object        
 3   created        675 non-null    int64         
 4   nid            675 non-null    int64         
 5   article_title  675 non-null    object        
 6   article_url    675 non-null    object        
 7   info_index     675 non-null    int64         
 8   datetime       675 non-null    object        
 9   datetime_my    675 non-null    object        
 10  date           675 non-null    datetime64[ns]
dtypes: datetime64[ns](1), int64(3), object(7)
memory usage: 58.1+ KB


None

,created,nid,info_index,date
count,6.750000e+02,675.000000,675.000000,675
mean,1.580411e+12,484909.220741,8.928889,2020-01-30 11:35:28
min,1.414764e+12,167488.000000,0.000000,2014-10-31 00:00:00
25%,1.478140e+12,313049.000000,1.000000,2016-11-02 12:00:00
50%,1.569859e+12,480608.000000,6.000000,2019-09-30 00:00:00
75%,1.666929e+12,641688.000000,12.000000,2022-10-27 12:00:00
max,1.789398e+12,818088.000000,67.000000,2026-09-14 00:00:00
std,1.111914e+11,191908.877350,10.200700,NaN


In [4]:
from transformers import pipeline
finbert = pipeline("sentiment-analysis",model="ProsusAI/finbert")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [5]:
def analyze_sentiment(text):

    text = str(text)
    text = " ".join(text.split())

    results = finbert(
        text,
        top_k=None,
        truncation=True
    )

    scores = {
        item["label"]: item["score"]
        for item in results
    }

    sentiment = max(scores,key=scores.get)

    sentiment_score = (
        scores["positive"]
        - scores["negative"]
    )

    return {
        "sentiment": sentiment,
        "positive_score": scores["positive"],
        "negative_score": scores["negative"],
        "neutral_score": scores["neutral"],
        "sentiment_score": sentiment_score
    }

In [6]:
from tqdm import tqdm
for index, row in tqdm(df.iterrows()):
    text = row["text"]
    sentiment_results = analyze_sentiment(text)
    df.at[index, "sentiment"] = sentiment_results["sentiment"]
    df.at[index, "positive_score"] = sentiment_results["positive_score"]
    df.at[index, "negative_score"] = sentiment_results["negative_score"]
    df.at[index, "neutral_score"] = sentiment_results["neutral_score"]
    df.at[index, "sentiment_score"] = sentiment_results["sentiment_score"]
    

675it [01:12,  9.37it/s]


In [7]:
df[
    [
        "company",
        "sentiment",
        "positive_score",
        "negative_score",
        "neutral_score",
        "sentiment_score"
    ]
].head()

,company,sentiment,positive_score,negative_score,neutral_score,sentiment_score
0,Petronas Chemicals,positive,0.944639,0.023755,0.031606,0.920885
1,Maybank,positive,0.561400,0.009733,0.428867,0.551667
2,Public Bank,neutral,0.033205,0.019832,0.946963,0.013373
3,CIMB,positive,0.666734,0.315408,0.017858,0.351326
4,Tenaga Nasional,negative,0.029521,0.960594,0.009884,-0.931073


In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")

df["n_tokens"] = df["text"].apply(
    lambda x: len(tokenizer(str(x),add_special_tokens=True)["input_ids"])
)

df["n_tokens"].describe()

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1236 > 512). Running this sequence through the model will result in indexing errors


count     675.000000
mean       92.364444
std        65.821496
min        24.000000
25%        53.000000
50%        76.000000
75%       117.000000
max      1236.000000
Name: n_tokens, dtype: float64

In [9]:
(df["n_tokens"] > 512).sum()

np.int64(1)

In [10]:
df["sentiment"].value_counts(normalize=True) * 100

sentiment
positive    50.074074
neutral     25.185185
negative    24.740741
Name: proportion, dtype: float64

In [11]:
pd.crosstab(
    df["company"],
    df["sentiment"]
)

sentiment,negative,neutral,positive
company,,,
CIMB,49,55,96
Maybank,35,39,75
Petronas Chemicals,13,4,15
Public Bank,25,11,48
Tenaga Nasional,45,61,104


In [12]:
df.groupby("company")["sentiment_score"].mean().sort_values(
    ascending=False
)

company
Tenaga Nasional       0.257265
Public Bank           0.244806
Maybank               0.243065
CIMB                  0.205983
Petronas Chemicals    0.028333
Name: sentiment_score, dtype: float64

In [13]:
df.to_csv(
    "../data/preprocessed_data/news_data_sentiment.csv",
    index=False
)

df.to_parquet(
    "../data/preprocessed_data/news_data_sentiment.parquet",
    index=False
)